# 🌳 Scaffold splitters: chemotype-aware generalization tests

Welcome! This notebook walks through chemsplit's **`scaffold`** family: six splitters that all group records by some notion of chemical *scaffold* — a Murcko core, a generic framework, a scaffold-tree node, a shared ring system, a matched-molecular-pair series, or an activity cliff — so that structurally related molecules land on the same side of the split. This is the standard defence against scaffold-level information leakage: a model that only memorises nearest-neighbour chemistry will fail here even if it looks great under a random split.

**Contents**
1. [📦 Setup & shared dataset](#1)
2. [🌳 The six splitters](#2)
   - [🧬 MurckoScaffoldSplitter](#2.1)
   - [🧪 GenericScaffoldSplitter](#2.2)
   - [🌲 ScaffoldTreeSplitter](#2.3)
   - [💍 RingSystemSplitter](#2.4)
   - [🔗 MatchedMolecularSeriesSplitter](#2.5)
   - [⚡ ActivityCliffSplitter](#2.6)

<a id="1"></a>
## 1. 📦 Setup & shared dataset

All classes below share one 150-molecule synthetic pool spanning 10 designed scaffold families (`chemsplit.datasets.make_scaffold_families`), plus a dedicated 80-molecule potency fixture with designed activity cliffs (`make_activity_cliffs`) for the last splitter. A small `summarize()` helper prints the size of each partition and a couple of interesting metadata fields so we don't have to repeat that boilerplate for every class.

In [1]:
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

import numpy as np

from chemsplit.datasets import make_activity_cliffs, make_scaffold_families
from chemsplit.splitters.scaffold import (
    ActivityCliffSplitter,
    GenericScaffoldSplitter,
    MatchedMolecularSeriesSplitter,
    MurckoScaffoldSplitter,
    RingSystemSplitter,
    ScaffoldTreeSplitter,
)


def summarize(result):
    print(f"{result.splitter_id}: n={result.n_records} "
          f"train={result.train.size} valid={result.valid.size} "
          f"test={result.test.size} discard={result.discard.size}")
    if result.groups is not None:
        print(f"  n_groups={len(set(result.groups.tolist()))}")
    interesting = {k: v for k, v in result.metadata.items()
                   if k in ("n_groups", "scaffold_smiles", "level", "n_cliff_pairs", "boundary_discarded")}
    for k, v in interesting.items():
        if isinstance(v, (list, np.ndarray)) and len(v) > 5:
            print(f"  {k}: [{len(v)} values, e.g. {list(v)[:3]}...]")
        else:
            print(f"  {k}: {v}")

In [2]:
fx = make_scaffold_families(n_scaffolds=10, per_scaffold=15, seed=0)
smiles = fx.smiles
print(f"{len(smiles)} molecules across {len(set(fx.groups_true.tolist()))} true scaffold families")

150 molecules across 10 true scaffold families


<a id="2"></a>
## 2. 🌳 The six splitters

<a id="2.1"></a>
### 2.1 🧬 MurckoScaffoldSplitter — the default comparability baseline

Groups records by Murcko scaffold. Deterministic without a seed (unless `group_assignment="random"`), `O(n)`, and needs no distance matrix — the default comparability baseline in the scaffold-split literature.

| Parameter | Meaning |
|---|---|
| `train_size`, `test_size` | target partition fractions |
| `random_state` | seed for tie-breaking / `group_assignment="random"` |
| `on_empty_scaffold` | what to do with acyclic molecules (no ring -> no scaffold): `"own_group"`, `"shared_group"`, `"discard"`, or `"raise"` |

In [3]:
sp = MurckoScaffoldSplitter(train_size=0.7, test_size=0.3, random_state=0, on_empty_scaffold="discard")
[result] = sp.split_result(smiles)
summarize(result)

murcko_scaffold: n=150 train=105 valid=0 test=45 discard=0
  n_groups=10
  n_groups: 10
  scaffold_smiles: [10 values, e.g. ['c1ccccc1', 'c1ccncc1', 'c1ccc2ccccc2c1']...]


> 💡 **Advantages**
> - Seed-free, `O(n)`, needs no distance matrix, and scales to millions of molecules.
> - The group key is a chemist-readable SMILES, so a disputed assignment can be inspected by eye — no clustering split offers that.
> - Harder than random and far cheaper than any similarity split, making it the default comparability baseline in the literature.
> - Directly targets the "new chemotype" question when a dataset genuinely spans many distinct frameworks.
>
> ⚠️ **Pitfalls**
> - **Systematically weaker than it looks.** A benzene-to-pyridine swap in a fused system lands in a different Murcko scaffold, so cross-boundary Tanimoto similarity routinely stays above 0.6 — confirm with `audit.nn_similarity_profile`.
> - On large diverse sets, singletons can dominate the scaffold distribution, degenerating the split toward random for most of the data (`metadata["singleton_frac"]`).
> - On a focused project set, one scaffold can hold most of the data, making the requested ratio unreachable — the splitter warns rather than fails.
> - The default `greedy_desc` ordering systematically enriches the test set in rare chemotypes; `group_assignment="random"` removes this bias at the cost of more variance.
> - Side chains are discarded entirely, so molecules with the same core but very different substituents land in the same group.
> - `on_empty_scaffold="shared_group"` silently lumps all acyclic molecules into one huge group.

<a id="2.2"></a>
### 2.2 🧪 GenericScaffoldSplitter — coarser, heteroatom-blind grouping

Groups records by generic (heteroatoms->carbon) scaffold framework — a coarser, more aggressive grouping than Murcko that collapses benzene/pyridine/pyrimidine analogues onto one topology.

| Parameter | Meaning |
|---|---|
| `train_size`, `test_size` | target partition fractions |
| `random_state` | seed for tie-breaking |
| `on_empty_scaffold` | handling of acyclic molecules, same options as above |

In [4]:
sp = GenericScaffoldSplitter(train_size=0.7, test_size=0.3, random_state=0, on_empty_scaffold="discard")
[result] = sp.split_result(smiles)
summarize(result)

generic_scaffold: n=150 train=105 valid=0 test=45 discard=0
  n_groups=5
  n_groups: 5


> 💡 **Advantages**
> - Collapses heteroatom variants onto one topology, closing the biggest leak in `murcko_scaffold`.
> - Produces fewer, larger groups, making the test set genuinely novel in ring topology.
> - Still seed-free and `O(n)`.
>
> ⚠️ **Pitfalls**
> - Groups get coarse enough that on small datasets the achievable train/test ratio drifts badly — expect `SizeToleranceWarning`.
> - Higher metric variance, since a single large group landing in test can dominate the score.

<a id="2.3"></a>
### 2.3 🌲 ScaffoldTreeSplitter — tunable pruning depth

Groups records by scaffold-tree node at a chosen pruning `level` (`level=0` is exactly `MurckoScaffoldSplitter`). Higher levels remove peripheral rings one at a time, giving a tunable strictness dial with actual chemical meaning.

| Parameter | Meaning |
|---|---|
| `level` | how many peripheral-ring pruning steps to apply (`0` = plain Murcko) |
| `prune_rule` | which ring to remove at each step, e.g. `"scaffold_tree"` |
| `train_size`, `test_size`, `random_state`, `on_empty_scaffold` | as above |

In [5]:
sp = ScaffoldTreeSplitter(level=1, prune_rule="scaffold_tree", train_size=0.7, test_size=0.3, random_state=0, on_empty_scaffold="discard")
[result] = sp.split_result(smiles)
summarize(result)

scaffold_tree: n=150 train=105 valid=0 test=45 discard=0
  n_groups=10
  n_groups: 10
  level: 1


> 💡 **Advantages**
> - Keeps a whole scaffold lineage together, so a test compound can't be a ring-truncated relative of a training compound — a leak plain Murcko splitting misses entirely.
> - The `level` knob gives a tunable strictness dial with actual chemical meaning, unlike an abstract distance cutoff.
> - Deterministic and seed-free.
>
> ⚠️ **Pitfalls**
> - `level` silently controls difficulty — always report it (it's written into `metadata` and `SplitResult.params`).
> - Ring pruning isn't canonical across toolkits, making cross-library comparison unsafe.
> - High `level` collapses everything into a handful of one-ring keys, giving huge groups and unreachable size targets.
> - Bridged, spiro, and macrocyclic systems frequently trigger `prune_failures`.

<a id="2.4"></a>
### 2.4 💍 RingSystemSplitter — shared individual ring systems

Groups records by shared individual ring system(s), transitively unioned via Union-Find. `linkage="any_shared"` unions any two molecules sharing >= 1 ring-system key; `"all_shared"` instead requires identical ring-system content.

| Parameter | Meaning |
|---|---|
| `linkage` | `"any_shared"` (transitive union) or `"all_shared"` (identical ring-system multiset) |
| `train_size`, `test_size`, `random_state`, `on_empty_scaffold` | as above |

In [6]:
sp = RingSystemSplitter(linkage="any_shared", train_size=0.7, test_size=0.3, random_state=0, on_empty_scaffold="discard")
[result] = sp.split_result(smiles)
summarize(result)

ring_system: n=150 train=105 valid=0 test=45 discard=0
  n_groups=10
  n_groups: 10


> 💡 **Advantages**
> - Targets novel ring chemistry directly — the exact claim behind most "scaffold hopping" results.
> - `any_shared` linkage catches a leak framework-level splits can't see: two molecules with different overall scaffolds sharing one highly characteristic ring system.
> - `csk` and `ring_size_profile` give progressively coarser, stricter variants without changing the algorithm.
>
> ⚠️ **Pitfalls**
> - With `any_shared` linkage, ubiquitous rings (benzene, pyridine, piperazine) can merge most of the dataset into one component — check `metadata["largest_component_frac"]`.
> - Transitive grouping isn't a distance — two molecules in the same group can be entirely dissimilar, linked only through a chain of shared rings.
> - Excluding common rings to avoid the giant component changes what the split means.
> - Spiro-merging policy affects group identity — fixed here at >=1 shared atom, but other toolkits differ.

<a id="2.5"></a>
### 2.5 🔗 MatchedMolecularSeriesSplitter — analogue-series isolation

Fragments every molecule on up to `max_cuts` acyclic single bonds via RDKit's MMPA; a constant context shared by >= `min_series_size` distinct molecules defines a series, unioned via Union-Find. This is the closest available test of "did the model learn the SAR, or memorise the series?".

| Parameter | Meaning |
|---|---|
| `min_series_size` | minimum number of molecules sharing a constant context to count as a series |
| `train_size`, `test_size`, `random_state`, `on_empty_scaffold` | as above |

In [7]:
sp = MatchedMolecularSeriesSplitter(min_series_size=2, train_size=0.7, test_size=0.3, random_state=0, on_empty_scaffold="discard")
[result] = sp.split_result(smiles)
summarize(result)

matched_molecular_series: n=150 train=105 valid=0 test=45 discard=0
  n_groups=20
  n_groups: 20
  boundary_discarded: 0


> 💡 **Advantages**
> - The closest available test of "did the model learn the SAR, or memorise the series?" — removes exactly the analogue pairs that make a nearest-neighbour baseline look competitive.
> - Chemically interpretable — every group can be explained as "these share a constant context".
> - `discard_boundary` mode keeps a familiar Murcko-split size profile while surgically removing the analogue leak.
>
> ⚠️ **Pitfalls**
> - Enumeration cost is the real constraint — `max_cuts>1` on sets above ~50,000 molecules is impractical by design.
> - Congeneric libraries collapse into one enormous group, making the requested ratio unreachable — check `largest_group_frac`.
> - MMP detection is sensitive to `min_constant_heavy_atoms` — the default of 5 is a convention, not a rule.
> - `discard_boundary` throws away exactly the records nearest the boundary, biasing the remaining test set to be harder than a uniform sample.

<a id="2.6"></a>
### 2.6 ⚡ ActivityCliffSplitter — a diagnostic, not a general-purpose split

Deliberately places activity-cliff compounds (similar structure, very different potency) into the test set and tags them for separate scoring. Unlike the other five, this is **not group-forming** — it's a diagnostic overlay on top of a regular split, isolating the failure mode that matters most in lead optimisation.

| Parameter | Meaning |
|---|---|
| `similarity_threshold` | minimum pairwise similarity to call two molecules "structurally similar" |
| `fold_change_threshold` | minimum potency fold-change (on the `y_scale`) to call the pair a "cliff" |
| `y_scale` | `"log"` or `"linear"` — must match how `y` was measured for the fold-change math to be meaningful |
| `train_size`, `test_size`, `random_state` | as above |

In [8]:
fx_cliff = make_activity_cliffs(n_pairs=40, seed=0)
sp = ActivityCliffSplitter(
    similarity_threshold=0.7, fold_change_threshold=3.0, y_scale="log",
    train_size=0.7, test_size=0.3, random_state=0,
)
[result] = sp.split_result(fx_cliff.smiles, y=fx_cliff.y)
summarize(result)
print("n_cliff_pairs:", result.metadata["n_cliff_pairs"])

activity_cliff: n=80 train=23 valid=0 test=57 discard=0
  n_cliff_pairs: 1358
n_cliff_pairs: 1358


Every molecule in this fixture shares one Murcko scaffold (only the substituent varies), so structural similarity is high across the *whole* set, not just within each of the 40 designed pairs — many cross-pair comparisons also clear the similarity/fold-change bar, which is why `n_cliff_pairs` (1358) is so much larger than 40. `ActivityCliffSplitter` prioritises pulling every cliff-tagged compound into test over honouring `train_size` exactly, which is also why train ends up well below the requested 70%. Both are expected behaviour, not a bug.

> 💡 **Advantages**
> - Isolates the failure mode that matters most in lead optimisation: two nearly identical molecules with very different potency.
> - `cliff_mask` lets the caller report cliff and non-cliff performance separately — the comparison, not the aggregate, is the point.
> - Four independent definitions of "structurally similar" let you check a conclusion isn't just a fingerprint artefact.
>
> ⚠️ **Pitfalls**
> - This is a **diagnostic, not a general-purpose split**. Report both cliff-set and aggregate performance — either alone is misleading.
> - Almost every descriptor-based model collapses to near-random on cliffs — that's the expected result, not a bug.
> - Cliff detection is extremely sensitive to `similarity_threshold`/`fold_change_threshold`.
> - Needs `y` on a log scale for the fold-change semantics to be meaningful; the implementation can't check this.
> - Assay noise can manufacture fake cliffs — aggregate replicates first and prefer single-assay data.

That covers all six `scaffold` splitters. They differ in *what* counts as a shared scaffold and how strictly it's enforced, but they all answer the same underlying question: will the model still work on a genuinely new chemotype, or did it just memorise the training analogues?